# ✋ ArSL V5 — تجربة **جديدة فعلياً** لتحسين دقة التصنيف

**النتيجة المرجعية V3:** دقة اختبار تاريخية 84.68% (630/744)، ودقة تحقق 86.56% (644/744). **لم تثبت بعد أي نتيجة لـ V5.**

بدلاً من إعادة Fine-Tuning أو إعادة تجربة TTA وأوزان الدمج:

1. نستخرج **ميزات (Embeddings)** من الطبقة قبل الأخيرة في MobileNetV2 وEfficientNetV2 المحفوظين.
2. ندرّب مصنفين مختلفين لهذه الميزات: **SVM/RBF** و **KNN**.
3. نختار المصنف باستخدام **جزء داخلي من صور التدريب فقط** (ليس Test ولا Validation).
4. نقارن المصنف الجديد ودمجه مع V3 على Validation. نحتفظ بـV3 إذا لم نربح **6 صور على الأقل من 744**.
5. نُجري اختباراً واحداً استكشافياً بعد تثبيت الاختيار. **هذا ليس اختباراً مستقلاً جديداً** لأن بيانات Test استُخدمت في نسخ سابقة.

**مهم:** لا نعدل النموذجين الأصليين، ولا نحتاج تدريب الشبكات العصبية مجدداً. يعتمد التشغيل على بياناتك الحالية وحزم نتائجك. لا يمكن ضمان تحسن الدقة.

## 1) تهيئة Colab وحفظ النتائج
فعّل T4 GPU إن توفر؛ GPU هنا لاستخراج الميزات فقط، أما SVM/KNN فيعملان على CPU. إذا كان Colab متصلاً مسبقاً بالصور، حدد مسارها في المتغير `DATA_ROOT_OVERRIDE` قبل تنفيذ خلية تنزيل البيانات.

In [ ]:
%pip -q install kagglehub scikit-learn pandas pillow joblib gradio

In [ ]:
import os, json, zipfile, hashlib, shutil, gc, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
import kagglehub
import joblib
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from google.colab import drive, files
from IPython.display import display

SEED=42
np.random.seed(SEED);random.seed(SEED);tf.random.set_seed(SEED)
drive.mount('/content/drive')
BASE=Path('/content/drive/MyDrive')
V3=BASE/'ArSL_V3_Accuracy_Improvement'/'Final_ArSL_V3'
V5=BASE/'ArSL_V5_Feature_Classification';V5.mkdir(parents=True,exist_ok=True)
WORK=Path('/content/arsl_v5_work');WORK.mkdir(parents=True,exist_ok=True)
DATA_ROOT_OVERRIDE='' # مسار الصور في Drive /content إذا كانت موجودة، لتجنب التنزيل
BATCH=32
RUN_EXPLORATORY_TEST=True   # لن يستخدم الاختبار للاختيار، ويمكن تغييره إلى False
print('TensorFlow:',tf.__version__, '| GPU:',tf.config.list_physical_devices('GPU'))
print('كل نتائجنا ستُحفظ في:',V5)

## 2) استعادة النموذجين وملف التقسيم الثابت

تحتاج ملف V3 النهائي (إذا لم يكن موجوداً في Drive) وملف النتائج **الأول** `Arabic_Sign_DL_Results.zip` لأنه يحتوي `split_manifest.csv`. **لا تعِد تقسيم البيانات عشوائياً**؛ هذا يفسد المقارنة.

In [ ]:
required={'model_0.keras','model_1.keras','labels.json','inference_config.json','metrics_v3.json'}
if not all((V3/f).exists() for f in required):
    print('لم أعثر على نماذج V3 في Google Drive. اختر ArSL_V3_Final_Results.zip:')
    upload=files.upload()
    filename='ArSL_V3_Final_Results.zip'
    if filename not in upload:raise FileNotFoundError(filename)
    V3.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile('/content/'+filename) as z:
        assert required.issubset(z.namelist()), 'ملف V3 ناقص'
        for name in required:
            (V3/name).write_bytes(z.read(name))

metadata=json.loads((V3/'labels.json').read_text(encoding='utf-8'))
classes=metadata['class_names']; class_to_id={c:i for i,c in enumerate(classes)}
assert len(classes)==31 and len(set(classes))==31
meta_v3=json.loads((V3/'metrics_v3.json').read_text(encoding='utf-8'))
conf_v3=json.loads((V3/'inference_config.json').read_text(encoding='utf-8'))
assert conf_v3['mode']=='ensemble' and len(conf_v3['members'])==2
assert abs(sum(m['weight'] for m in conf_v3['members'])-1)<1e-8
print('نسخة V3 الأصلية موجودة. أسماء الفئات:',len(classes))

old_zip=BASE/'ArSL_Accuracy_Improvement'/'Arabic_Sign_DL_Results.zip'
if not old_zip.exists():old_zip=Path('/content/Arabic_Sign_DL_Results.zip')
if not old_zip.exists():
    print('ارفع الملف الأول Arabic_Sign_DL_Results.zip للحصول على التقسيم الثابت:')
    upload=files.upload()
    if 'Arabic_Sign_DL_Results.zip' not in upload:raise FileNotFoundError('Arabic_Sign_DL_Results.zip')
    old_zip=Path('/content/Arabic_Sign_DL_Results.zip')
with zipfile.ZipFile(old_zip) as z:
    assert 'split_manifest.csv' in z.namelist()
    (WORK/'split_manifest.csv').write_bytes(z.read('split_manifest.csv'))
manifest=pd.read_csv(WORK/'split_manifest.csv')
assert set(manifest.columns)>={'path','label_id','split'}
assert manifest.split.value_counts().to_dict()=={'train':3472,'val':744,'test':744}
manifest['key']=[(Path(str(p)).parent.name,Path(str(p)).name) for p in manifest.path]
assert not manifest.key.duplicated().any()
print('التقسيم ثابت:',manifest.split.value_counts().to_dict())

## 3) استعادة الصور وإعادة تكوين 6165 / 744 / 744

كود مطابق لسياسة استبعاد الصور البايتية المتطابقة التي استخدمت سابقاً؛ الصور التي لم تدخل التدريب الأول تدخل التدريب فقط. قد يستغرق تنزيل البيانات والفحص وقتاً، لكنه يتم مرة واحدة. إذا كان KaggleHub ما زال يحتفظ بالبيانات، غالباً لن يعيد تنزيلها.

In [ ]:
DATASET='muhammadalbrham/rgb-arabic-alphabets-sign-language-dataset'
root=Path(DATA_ROOT_OVERRIDE) if DATA_ROOT_OVERRIDE else Path(kagglehub.dataset_download(DATASET))
if not root.exists():raise FileNotFoundError(str(root))
def find_class_root(base):
    options=[base]+[p for p in base.rglob('*') if p.is_dir()]
    return max(options,key=lambda d:sum((d/c).is_dir() for c in classes))
class_root=find_class_root(root)
if sum((class_root/c).is_dir() for c in classes)!=31:
    archives=list(root.rglob('*.zip'))
    if not archives:raise RuntimeError('لم نجد مجلدات الفئات؛ اضبط DATA_ROOT_OVERRIDE')
    dest=WORK/'dataset_extracted';dest.mkdir(exist_ok=True)
    with zipfile.ZipFile(archives[0]) as z:z.extractall(dest)
    class_root=find_class_root(dest)
assert sum((class_root/c).is_dir() for c in classes)==31
exts={'.jpg','.jpeg','.png','.bmp'}
all_paths=sorted(p for c in classes for p in (class_root/c).iterdir() if p.is_file() and p.suffix.lower() in exts)
by_key={(p.parent.name,p.name):p for p in all_paths}
assert len(by_key)==len(all_paths)
missing=[k for k in manifest.key if k not in by_key]
if missing:raise FileNotFoundError(f'بيانات التقسيم القديم غير موجودة: {missing[:5]}')
for rec in manifest.itertuples(index=False):
    assert class_to_id[rec.key[0]]==int(rec.label_id),'التسميات غير متطابقة'

seen={hashlib.sha256(by_key[k].read_bytes()).hexdigest() for k in manifest.key}
old_keys=set(manifest.key)
extras=[]; duplicates=0; invalid=0
for path in all_paths:
    key=(path.parent.name,path.name)
    if key in old_keys:continue
    try:
        with Image.open(path) as im:im.verify()
        digest=hashlib.sha256(path.read_bytes()).hexdigest()
    except Exception:invalid+=1;continue
    if digest in seen:duplicates+=1;continue
    seen.add(digest);extras.append(path)

def split(name):
    frame=manifest.loc[manifest.split==name]
    return [str(by_key[k]) for k in frame.key],frame.label_id.to_numpy(np.int32)
train_paths,train_y=split('train'); val_paths,val_y=split('val'); test_paths,test_y=split('test')
train_paths.extend(str(p) for p in extras)
train_y=np.concatenate([train_y,np.array([class_to_id[p.parent.name] for p in extras],np.int32)])
assert (len(train_y),len(val_y),len(test_y))==(6165,744,744)
assert not (set(train_paths)&set(val_paths) or set(train_paths)&set(test_paths) or set(val_paths)&set(test_paths))
print('صور التدريب/التحقق/الاختبار:',len(train_y),len(val_y),len(test_y))
print('مضاف للتدريب فقط:',len(extras),'| مستبعدة مكررة:',duplicates,'| تالفة:',invalid)
print('تنبيه: الفصل هنا على مستوى الصورة وليس على مستوى الشخص.')

## 4) استخراج ميزات عميقة دون إعادة تدريب CNN

نأخذ متجه الطبقة `global_average_pooling2d` **قبل** طبقة التصنيف لكل شبكة. في مرور واحد لكل صورة نحصل أيضاً على توقعات النموذج الأساسية. نخزّن الميزات مؤقتاً في Google Drive لإعادة استخدام الخلية دون إعادة الحساب.

**ضابط صحة مهم:** يجب أن تعيد توقعات V3 على Validation نتيجة **644/744**؛ إذا اختلفت نتوقف حتى لا نبني تحسناً وهمياً.

In [ ]:
AUTOTUNE=tf.data.AUTOTUNE

def image_ds(paths,labels,size):
    ds=tf.data.Dataset.from_tensor_slices((list(map(str,paths)),np.asarray(labels,dtype=np.int32)))
    def parse(path,y):
        im=tf.io.decode_image(tf.io.read_file(path),channels=3,expand_animations=False)
        im.set_shape((None,None,3))
        x=tf.image.resize_with_pad(tf.cast(im,tf.float32),*size)
        return x,y
    return ds.map(parse,num_parallel_calls=AUTOTUNE).batch(BATCH).prefetch(AUTOTUNE)

def data_signature():
    h=hashlib.sha256(b'arsl_v5_feature_1')
    for m in conf_v3['members']:
        h.update(hashlib.sha256((V3/m['file']).read_bytes()).digest())
    for p,y in zip(train_paths,train_y):
        h.update(f'{Path(p).parent.name}/{Path(p).name}|{int(y)}\n'.encode())
    for p,y in zip(val_paths,val_y):
        h.update(f'V/{Path(p).parent.name}/{Path(p).name}|{int(y)}\n'.encode())
    return h.hexdigest()
SIGNATURE=data_signature()
CACHE=V5/'feature_cache_v5.npz'
if CACHE.exists():
    try:
        c=np.load(CACHE,allow_pickle=False)
        if str(c['signature'].item())!=SIGNATURE:raise ValueError('بصمة ملفات البيانات أو النماذج تغيرت')
        features_train=[c['train_0'],c['train_1']]
        features_val=[c['val_0'],c['val_1']]
        val_probs=[c['prob_0'],c['prob_1']]
        assert len(features_train[0])==6165 and len(features_val[0])==744
        print('تم تحميل الميزات المحفوظة في Drive. لا حاجة لتكرار استخراج CNN.')
        cached=True
    except Exception as e:
        print('إهمال ذاكرة ميزات قديمة:',e);cached=False
else:cached=False

if not cached:
    features_train=[];features_val=[];val_probs=[]
    for i,member in enumerate(conf_v3['members']):
        mdl=tf.keras.models.load_model(V3/member['file'],compile=False)
        assert tuple(mdl.input_shape[1:3])==tuple(member['size'])
        extractor=tf.keras.Model(mdl.inputs,[mdl.get_layer('global_average_pooling2d').output,mdl.outputs[0]])
        print(f'استخرج ميزات الشبكة {i+1}/2 من Train...')
        f_train,_=extractor.predict(image_ds(train_paths,train_y,tuple(member['size'])),verbose=1)
        print(f'استخرج ميزات الشبكة {i+1}/2 من Validation...')
        f_val,p_val=extractor.predict(image_ds(val_paths,val_y,tuple(member['size'])),verbose=1)
        features_train.append(np.asarray(f_train,dtype=np.float32))
        features_val.append(np.asarray(f_val,dtype=np.float32))
        val_probs.append(np.asarray(p_val,dtype=np.float32))
        del mdl,extractor;tf.keras.backend.clear_session();gc.collect()
    np.savez_compressed(CACHE,signature=np.array(SIGNATURE),
                        train_0=features_train[0],train_1=features_train[1],
                        val_0=features_val[0],val_1=features_val[1],
                        prob_0=val_probs[0],prob_1=val_probs[1])
    print('حُفظت الميزات:',CACHE)

assert all(x.shape[0]==6165 for x in features_train)
assert all(x.shape[0]==744 for x in features_val)
base_val=(val_probs[0]+val_probs[1])/2
base_correct=int(np.sum(base_val.argmax(1)==val_y))
expected=int(meta_v3['validation_accuracy']*744+0.5)
print('إعادة إنتاج V3:',base_correct,'/',len(val_y),'| المتوقع:',expected)
if base_correct!=644 or expected!=644:
    raise RuntimeError('❌ لم تتطابق نتيجة V3؛ افحص معالجة الصور وترتيب الفئات قبل المتابعة.')

## 5) تدريب مصنّفَين على التمثيلات بدل صور RGB

**الفكرة:** كل صورة تصبح متجهين عدديين من الشبكتين، نطبّع كل متجه إلى طول 1 ثم نصل المتجهين. نضغط الأبعاد باستخدام PCA **بعد تعلّمه من بيانات التدريب فقط**.

نستخدم 15% من Train لاختيار SVM أو KNN، ثم نعيد تدريب المصنف الفائز على كامل Train. **لا نستخدم Validation أو Test لاتخاذ قرار أي المصنفين أفضل.**

In [ ]:
def fuse(a,b):
    return np.concatenate((normalize(a.astype('float32')),normalize(b.astype('float32'))),axis=1).astype('float32')
Xtr=fuse(*features_train)
Xval=fuse(*features_val)
tr_idx,inner_idx=train_test_split(np.arange(len(train_y)),test_size=0.15,random_state=SEED,stratify=train_y)
X_inner_train=Xtr[tr_idx]; X_inner_eval=Xtr[inner_idx]
y_inner_train=train_y[tr_idx];y_inner_eval=train_y[inner_idx]

inner_pca=PCA(n_components=128,svd_solver='randomized',random_state=SEED)
Zinner_train=inner_pca.fit_transform(X_inner_train)
Zinner_eval=inner_pca.transform(X_inner_eval)
models={
    'SVM_RBF':SVC(C=4.0,kernel='rbf',gamma='scale',probability=True,random_state=SEED,cache_size=400),
    'KNN_Cosine':KNeighborsClassifier(n_neighbors=9,weights='distance',metric='cosine'),
}
internal_rows=[]
for name,candidate in models.items():
    print('تدريب',name)
    candidate.fit(Zinner_train,y_inner_train)
    inner_predictions=candidate.predict(Zinner_eval)
    internal_rows.append({'method':name,'inner_accuracy':float(accuracy_score(y_inner_eval,inner_predictions)),
                          'inner_macro_f1':float(f1_score(y_inner_eval,inner_predictions,average='macro'))})
internal_table=pd.DataFrame(internal_rows).sort_values(['inner_accuracy','inner_macro_f1'],ascending=False)
display(internal_table)
chosen_method=internal_table.iloc[0]['method']
print('اختيار داخلي من TRAIN فقط:',chosen_method)

del models,inner_pca,Zinner_train,Zinner_eval
# إعادة ملاءمة الميزات والمصنّف على جميع صور Train، دون Val أو Test
reducer=PCA(n_components=128,svd_solver='randomized',random_state=SEED)
Ztr=reducer.fit_transform(Xtr)
Zval=reducer.transform(Xval)
if chosen_method=='SVM_RBF':
    specialist=SVC(C=4.0,kernel='rbf',gamma='scale',probability=True,random_state=SEED,cache_size=400)
else:
    specialist=KNeighborsClassifier(n_neighbors=9,weights='distance',metric='cosine')
specialist.fit(Ztr,train_y)
assert np.array_equal(specialist.classes_,np.arange(31))
spec_val=np.asarray(specialist.predict_proba(Zval),dtype=np.float32)
assert spec_val.shape==(744,31) and np.isfinite(spec_val).all()
print('تم تدريب',chosen_method,'على 6165 صورة. دقة المصنف منفرداً على Validation:',f'{accuracy_score(val_y,spec_val.argmax(1)):.2%}')

## 6) مقارنة محدودة على Validation والاحتفاظ بـV3 عند غياب تحسن واضح

هذه **طرق مختلفة** عن تعديل 50/50 بين الشبكتين مباشرة: هنا نضيف مصنفاً جديداً يعمل على الميزات. نختبر المصنف وحده ودمجه بنسبتين ثابتتين؛ لا نجري بحثاً واسعاً عن مئات الإعدادات.

المعيار: تحسن **6 صور صحيحة على الأقل** بالنسبة إلى V3. ولأن Validation استُخدمت سابقاً عدة مرات، فإن التحسن الحالي **استكشافي** ويجب التأكد منه بصور من أشخاص جدد.

In [ ]:
val_variants={
    'V3_original':base_val,
    'specialist_only':spec_val,
    'V3_75_specialist_25':0.75*base_val+0.25*spec_val,
    'V3_50_specialist_50':0.50*base_val+0.50*spec_val,
}
rows=[]
for name,p in val_variants.items():
    rows.append({'mode':name,'correct':int(np.sum(p.argmax(1)==val_y)),
                 'validation_accuracy':float(accuracy_score(val_y,p.argmax(1))),
                 'macro_f1':float(f1_score(val_y,p.argmax(1),average='macro'))})
validation_table=pd.DataFrame(rows).sort_values(['correct','macro_f1'],ascending=False)
display(validation_table)
champion=validation_table.iloc[0]
GAIN_MIN=6
mode=str(champion['mode']) if int(champion['correct']) >= base_correct+GAIN_MIN else 'V3_original'
accepted=mode!='V3_original'
chosen_val=val_variants[mode]
print('V3:',base_correct,'/',len(val_y), '=>',f'{base_correct/744:.2%}')
print('قرار V5:',mode,'| الصحيح:',int(np.sum(chosen_val.argmax(1)==val_y)),'/',len(val_y), '| اعتمد:',accepted)
print('نقطة مهمة: لا نختار شيئاً بناء على Test.')

class_rows=[]
for c,name in enumerate(classes):
    ix=np.flatnonzero(val_y==c)
    class_rows.append({'class':name,'arabic':metadata['arabic_labels'].get(name,name),
                       'samples':len(ix),'V3_correct':int((base_val[ix].argmax(1)==val_y[ix]).sum()),
                       'V5_correct':int((chosen_val[ix].argmax(1)==val_y[ix]).sum())})
class_table=pd.DataFrame(class_rows)
class_table['difference']=class_table.V5_correct-class_table.V3_correct
print('مقارنة الفئات الأكثر تأثراً:')
display(class_table.sort_values('difference').head(8))
display(class_table.sort_values('difference',ascending=False).head(8))

## 7) اختبار **استكشافي** بعد تثبيت قرار Validation

يعمل مرة واحدة فقط بعد تثبيت `mode`. **لا تُغيّر mode استجابةً لنتيجة الاختبار**. الصور الـ744 هنا ليست مستقلة تماماً عن عملية التطوير السابقة؛ لذلك نعرض الدقة للمقارنة التاريخية ولا نزعم تعميماً على أشخاص جدد.

يمكن تخطي الاختبار بتعيين `RUN_EXPLORATORY_TEST=False`.

In [ ]:
result_test=None
if RUN_EXPLORATORY_TEST and accepted:
    test_features=[];test_probs=[]
    for i,m in enumerate(conf_v3['members']):
        mdl=tf.keras.models.load_model(V3/m['file'],compile=False)
        extractor=tf.keras.Model(mdl.inputs,[mdl.get_layer('global_average_pooling2d').output,mdl.outputs[0]])
        f,p=extractor.predict(image_ds(test_paths,test_y,tuple(m['size'])),verbose=1)
        test_features.append(np.asarray(f,dtype=np.float32))
        test_probs.append(np.asarray(p,dtype=np.float32))
        del mdl,extractor;tf.keras.backend.clear_session();gc.collect()
    base_test=(test_probs[0]+test_probs[1])/2
    base_test_correct=int(np.sum(base_test.argmax(1)==test_y))
    assert base_test_correct==630, f'اختبار V3 لم يُستعد كما هو: {base_test_correct} بدلاً من 630'
    spec_test=specialist.predict_proba(reducer.transform(fuse(*test_features)))
    if mode=='specialist_only': p_test=spec_test
    elif mode=='V3_75_specialist_25':p_test=0.75*base_test+0.25*spec_test
    elif mode=='V3_50_specialist_50':p_test=0.50*base_test+0.50*spec_test
    else:raise RuntimeError('mode غير معروف')
    result_test={'accuracy':float(accuracy_score(test_y,p_test.argmax(1))),
                 'macro_f1':float(f1_score(test_y,p_test.argmax(1),average='macro')),
                 'correct':int(np.sum(p_test.argmax(1)==test_y)),
                 'samples':len(test_y),'status':'exploratory_reused_test'}
    print('V3 تاريخياً:',base_test_correct,'/',len(test_y),f'({base_test_correct/len(test_y):.2%})')
    print('V5 استكشافي:',result_test['correct'],'/',len(test_y),f"({result_test['accuracy']:.2%})")
    print('لا نعتمد التعديل أو نلغيه بناءً على هذه النتائج؛ القرار ثُبّت باستخدام Validation.')
else:
    print('لم يُجرَ اختبار جديد لأننا أبقينا V3، أو لأن RUN_EXPLORATORY_TEST=False.')

## 8) حفظ المصنف وملفات صفحة ويب متوافقة مع الطريقة الجديدة

ستُحفظ نسخة في Drive تشمل النموذجين الأصليين، مصنف SVM/KNN الإضافي، وإعداد اختيار V5. **إذا لم تُقبل V5، ستعمل صفحة الويب بنفس دمج V3 دون تغيير.**

In [ ]:
OUT=V5/'Final_V5_Deployment';OUT.mkdir(parents=True,exist_ok=True)
for name in ['model_0.keras','model_1.keras','labels.json']:
    if not (OUT/name).exists() or (OUT/name).stat().st_size!=(V3/name).stat().st_size:
        shutil.copy2(V3/name,OUT/name)
joblib.dump({'reducer':reducer,'classifier':specialist,'method':chosen_method},OUT/'v5_classifier.joblib',compress=3)
config={'mode':mode,'accepted_on_validation':accepted,'class_count':31,
        'image_sizes':[m['size'] for m in conf_v3['members']],
        'reference_v3_weights':[m['weight'] for m in conf_v3['members']],
        'input':'RGB 0..255, resize_with_pad; CNN internal preprocessing',
        'warning':'score is not a guarantee for unknown people; probability estimates can be imperfect'}
(OUT/'inference_config.json').write_text(json.dumps(config,ensure_ascii=False,indent=2),encoding='utf-8')
results={'V3_historical_test_accuracy':float(meta_v3['new_test_accuracy']),
         'V3_validation_accuracy':float(meta_v3['validation_accuracy']),
         'v5_validation_accuracy':float(accuracy_score(val_y,chosen_val.argmax(1))),
         'v5_validation_correct':int(np.sum(chosen_val.argmax(1)==val_y)),
         'baseline_validation_correct':base_correct,
         'inner_training_comparison':internal_rows,'classifier_method':chosen_method,
         'decision':mode,'accepted':accepted,'test_if_run':result_test,
         'limitations':['same image-level split, signer identities are not disjoint',
                        'existing validation/test were reused in prior projects; report as exploratory',
                        'a new person-disjoint set is necessary for trustworthy generalization']}
(OUT/'metrics_v5.json').write_text(json.dumps(results,ensure_ascii=False,indent=2),encoding='utf-8')
validation_table.to_csv(OUT/'validation_candidates.csv',index=False)
class_table.to_csv(OUT/'class_comparison.csv',index=False)
(OUT/'requirements.txt').write_text('tensorflow>=2.16,<3\nnumpy>=1.26\nscikit-learn>=1.4\njoblib>=1.3\nPillow>=10\ngradio>=5,<7\n',encoding='utf-8')
print('حُفظت نتائج V5:',OUT)

## 9) إنشاء تطبيق Gradio ثم حزمة ZIP

يمكن تنزيل الحزمة وفتح صفحة الويب **بعد نجاح التدريب والاختيار**. إذا انقطع Colab قبل هذه المرحلة، ما تزال ميزات V5 مخزنة في Drive ويمكنك إعادة تشغيل الخلايا الأخيرة.

In [ ]:
APP = '''"""ArSL V5 -- ensemble or learned-feature classifier, images only."""
from pathlib import Path
import json, numpy as np, tensorflow as tf, joblib
from PIL import Image, ImageOps
from sklearn.preprocessing import normalize
import gradio as gr

ROOT=Path(__file__).resolve().parent
labels=json.loads((ROOT/'labels.json').read_text(encoding='utf-8'))
classes=labels['class_names']
arabic=labels.get('arabic_labels',{})
config=json.loads((ROOT/'inference_config.json').read_text(encoding='utf-8'))
models=[tf.keras.models.load_model(ROOT/f'model_{i}.keras',compile=False) for i in range(2)]
extractors=[tf.keras.Model(m.inputs,[m.get_layer('global_average_pooling2d').output,m.outputs[0]]) for m in models]
learned=joblib.load(ROOT/'v5_classifier.joblib')
assert all(m.output_shape[-1]==len(classes) for m in models)

def classify(image):
    if image is None:return {},'ارفع صورة لإشارة يد واضحة.'
    rgb=np.asarray(ImageOps.exif_transpose(image).convert('RGB'),dtype=np.float32)
    features=[];predictions=[]
    for e,size in zip(extractors,config['image_sizes']):
        im=tf.image.resize_with_pad(tf.convert_to_tensor(rgb),int(size[0]),int(size[1]))
        feat,probs=e.predict(tf.expand_dims(im,0),verbose=0)
        features.append(feat);predictions.append(probs[0])
    baseline=0.5*predictions[0]+0.5*predictions[1]
    if config['mode']!='V3_original':
        fv=np.concatenate([normalize(f.astype(np.float32)) for f in features],axis=1)
        spec=learned['classifier'].predict_proba(learned['reducer'].transform(fv))[0]
        if config['mode']=='specialist_only':prediction=spec
        elif config['mode']=='V3_75_specialist_25':prediction=0.75*baseline+0.25*spec
        elif config['mode']=='V3_50_specialist_50':prediction=0.5*baseline+0.5*spec
        else:raise ValueError('غير معروف: '+str(config['mode']))
    else:prediction=baseline
    top=np.argsort(prediction)[::-1][:5]
    answer={f'{arabic.get(classes[int(i)],classes[int(i)])} | {classes[int(i)]}':float(prediction[i]) for i in top}
    note='النتائج تقديرات على صور حروف ثابتة فقط، وليست ترجمة للغة الإشارة في الفيديو.'
    return answer,note

app=gr.Interface(
    fn=classify,
    inputs=gr.Image(type='pil',sources=['upload','webcam'],label='ارفع أو التقط صورة اليد'),
    outputs=[gr.Label(label='أقرب 5 حروف',num_top_classes=5),gr.Textbox(label='ملاحظة')],
    title='ArSL — التعرف على حروف لغة الإشارة العربية',
    description='V5: نموذجَا CNN مع مصنف ميزات (إن اجتاز التحسن شرط القبول).'
)
if __name__=='__main__':app.launch(share=True)
'''
(OUT/'app.py').write_text(APP,encoding='utf-8')
archive=shutil.make_archive('/content/ArSL_V5_Final_Results','zip',OUT)
print('حزمة V5:',archive,'\nحجمها MB:',round(Path(archive).stat().st_size/1024**2,2))
print('قرار التحسين:',config['mode'],'| تم القبول:',config['accepted_on_validation'])
print('الحزمة محفوظة أيضاً في Drive:',OUT)
# اختياري: نفذ السطر التالي فقط إذا أردت تحميل الحزمة على جهازك الآن:
# files.download(archive)

## 10) تشغيل صفحة الويب في نفس جلسة Colab (اختياري)

بمجرد ظهور `https://....gradio.live` يمكن فتحه من الهاتف. لا يحتاج إعادة تدريب. رابط Gradio مؤقت ويعمل ما دامت جلسة Colab نشطة.

In [ ]:
import importlib.util
spec=importlib.util.spec_from_file_location('arsl_v5_live',OUT/'app.py')
webapp=importlib.util.module_from_spec(spec)
spec.loader.exec_module(webapp)
webapp.app.launch(share=True,debug=False)

## ملخص علمي للمناقشة

- **Feature Extraction:** استخدمنا الطبقة قبل الأخيرة من CNN كواصف عددي بدلاً من بناء CNN ثالثة.
- **SVM:** يحاول إنشاء حدود فصل غير خطية باستخدام RBF، و**KNN:** يصنف الصورة وفق تشابهها مع صور التدريب في فضاء الميزات.
- **PCA:** يضغط آلاف الأبعاد إلى 128 بُعداً ويُعلَّم على Train فقط.
- **التعميم:** الفرق بين اختبار صور عشوائية واختبار أشخاص جدد قد يكون كبيراً. النتائج الماضية لا تثبت دقة 84.68% على أشخاص لم يظهروا في البيانات.
- **قبول التحسين:** نختار المصنف داخل Train، ونقيّم فائدته على Validation ثم نعرض Test استكشافياً مرة واحدة. لقد استخدمت Validation وTest عدة مرات في إصدارات سابقة؛ يلزم اختبار جديد مستقل لتقييم نهائي موثوق.
- **إذا لم تتحسن الدقة:** نقبل النتيجة ونحافظ على V3، لأن إعادة التجارب بلا دليل ليست تقدماً.

**مراجع المنهج:** [Keras Feature Extraction](https://keras.io/getting_started/faq/) — [Scikit-learn SVC](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html)